# 01 — Exploratory Data Analysis

Understand the raw Sparkov dataset before any label derivation or modelling.

**Outputs:**
- Schema validation and missing-value audit
- Distribution of `amt`, `category`, `state`, `gender`, `job`
- Class imbalance: `is_fraud` prevalence overall and by category/state
- Temporal coverage: transaction volume over time
- Cardholder and merchant count sanity checks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

In [ ]:
# Load — adjust paths if you placed train/test CSVs differently
train = pd.read_csv('../data/raw/fraudTrain.csv', parse_dates=['trans_date_trans_time', 'dob'])
test  = pd.read_csv('../data/raw/fraudTest.csv',  parse_dates=['trans_date_trans_time', 'dob'])

print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')
train.head(3)

In [ ]:
# Schema and missing values
print(train.dtypes)
print('\nMissing values:')
print(train.isnull().sum())

In [ ]:
# Fraud prevalence
print('Overall fraud rate:', train['is_fraud'].mean().round(4))
print('\nFraud rate by category:')
print(train.groupby('category')['is_fraud'].mean().sort_values(ascending=False).round(4))

In [ ]:
# Amount distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train['amt'].hist(bins=80, ax=axes[0])
axes[0].set_title('Amount distribution')
axes[0].set_xlabel('amt (USD)')

np.log1p(train['amt']).hist(bins=80, ax=axes[1])
axes[1].set_title('log(1 + amt)')
plt.tight_layout()

In [ ]:
# Cardholder / merchant counts
print(f"Unique cardholders (train): {train['cc_num'].nunique():,}")
print(f"Unique merchants  (train): {train['merchant'].nunique():,}")
print(f"Unique categories (train): {train['category'].nunique()}")
print(f"Date range: {train['trans_date_trans_time'].min()} → {train['trans_date_trans_time'].max()}")

In [ ]:
# Transaction volume over time
daily = train.resample('D', on='trans_date_trans_time')['is_fraud'].agg(['sum', 'count'])
daily.columns = ['fraud_count', 'total_count']

fig, ax = plt.subplots(figsize=(14, 4))
daily['total_count'].plot(ax=ax, label='Total')
daily['fraud_count'].plot(ax=ax, label='Fraud', color='crimson')
ax.set_title('Daily transaction volume')
ax.legend()
plt.tight_layout()